# Homework: Vector Search

Text is turned into vectors and searched by similarity, then vector search is
combined with keyword search. No RAG — the focus is search only.

The knowledge base is the **course lessons themselves**: 72 markdown pages pulled
from the course repo, pinned to commit `8c1834d`.

Embeddings use the lightweight **ONNX Embedder** (`Xenova/all-MiniLM-L6-v2`)
instead of `sentence-transformers`. Both produce identical vectors, so the answers
match either way.

## Setup

The `Embedder` wraps the ONNX model downloaded by `download.py`. `encode` returns a
single normalized 384-dim vector; `encode_batch` returns one per text.

In [1]:
from embedder import Embedder

embedder = Embedder()

## Load the data

Lesson pages pulled straight from the course repo. Each document is a dict with
`filename` and `content`; there are 72 pages.

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print("Pages loaded:", len(documents))

Pages loaded: 72


## Q1. Embedding a query

Embed the ANN query and read the first component of the resulting vector.

In [3]:
ann_query = "How does approximate nearest neighbor search work?"
ann_query_vector = embedder.encode(ann_query)

print("Vector length:", len(ann_query_vector))
print("Q1 — v[0]:", ann_query_vector[0])

Vector length: 384
Q1 — v[0]: -0.020582034180917443


**Answer Q1: ~-0.02**

## Q2. Cosine similarity

Normalized vectors, so the dot product is the cosine similarity. Embed the content
of `07-sqlitesearch-vector.md` and compare it with the Q1 query vector.

In [4]:
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"
page = next(d for d in documents if d["filename"] == target)

page_vector = embedder.encode(page["content"])
similarity = ann_query_vector.dot(page_vector)

print("Q2 — cosine similarity:", similarity)

Q2 — cosine similarity: 0.36107008472347096


**Answer Q2: ~0.37**

## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding. Split pages
into overlapping chunks, embed each chunk, stack the vectors into a matrix, and
score the Q1 query against every chunk.

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print("Chunks:", len(chunks))
# Each chunk keeps filename and content, plus a start offset.
print("Chunk fields:", list(chunks[0].keys()))

Chunks: 295
Chunk fields: ['start', 'content', 'filename']


In [6]:
import numpy as np
from tqdm.auto import tqdm

# Embed chunk contents in batches; stack into one (n_chunks, 384) matrix.
chunk_texts = [chunk["content"] for chunk in chunks]
batch_size = 50

chunk_vectors = []
for i in tqdm(range(0, len(chunk_texts), batch_size)):
    batch = chunk_texts[i:i + batch_size]
    chunk_vectors.extend(embedder.encode_batch(batch))

chunk_vectors = np.array(chunk_vectors)
print("Matrix shape:", chunk_vectors.shape)

  0%|          | 0/6 [00:00<?, ?it/s]

Matrix shape: (295, 384)


In [7]:
# Score every chunk against the Q1 query and take the best one.
scores = chunk_vectors.dot(ann_query_vector)
best = int(np.argmax(scores))

print("Q3 — top chunk file:", chunks[best]["filename"])

Q3 — top chunk file: 02-vector-search/lessons/07-sqlitesearch-vector.md


**Answer Q3: `02-vector-search/lessons/07-sqlitesearch-vector.md`**

## Q4. Vector search with minsearch

Same vector search, but with `VectorSearch` from minsearch instead of doing the
dot product by hand. Index the chunk vectors and search a new query.

In [8]:
from minsearch import VectorSearch

vector_index = VectorSearch()
vector_index.fit(chunk_vectors, chunks)

In [9]:
metric_query = "What metric do we use to evaluate a search engine?"
metric_query_vector = embedder.encode(metric_query)

results = vector_index.search(metric_query_vector, num_results=5)
print("Q4 — first result:", results[0]["filename"])

Q4 — first result: 04-evaluation/lessons/05-search-metrics.md


**Answer Q4: `04-evaluation/lessons/05-search-metrics.md`**

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words. Index the same
chunks for text search, run both methods on the same query, and find the file that
shows up in the vector top-5 but not the text top-5.

In [10]:
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [11]:
pg_query = "How do I store vectors in PostgreSQL?"
pg_query_vector = embedder.encode(pg_query)

text_top5 = text_index.search(pg_query, num_results=5)
vector_top5 = vector_index.search(pg_query_vector, num_results=5)

text_files = {r["filename"] for r in text_top5}
vector_files = {r["filename"] for r in vector_top5}

print("Text results:  ", [r["filename"] for r in text_top5])
print("Vector results:", [r["filename"] for r in vector_top5])
print("Q5 — only in vector:", vector_files - text_files)

Text results:   ['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']
Vector results: ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']
Q5 — only in vector: {'02-vector-search/lessons/08-pgvector.md'}


**Answer Q5: `02-vector-search/lessons/08-pgvector.md`**

## Q6. Hybrid search

Combine both rankings with Reciprocal Rank Fusion (RRF): each document scores by
its position in each list, summed with a constant `k`. A document strong in both
lists ends up on top, even if it never leads either list on its own.

In [12]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [13]:
tools_query = "How do I give the model access to tools?"
tools_query_vector = embedder.encode(tools_query)

# Same top-5 depth for both methods before fusing.
text_results = text_index.search(tools_query, num_results=5)
vector_results = vector_index.search(tools_query_vector, num_results=5)

fused = rrf([vector_results, text_results])
print("Fused ranking:", [r["filename"] for r in fused])
print("Q6 — ranked first:", fused[0]["filename"])

Fused ranking: ['01-agentic-rag/lessons/13-function-calling.md', '01-agentic-rag/lessons/01-intro.md', '01-agentic-rag/lessons/14-agentic-loop.md', '04-evaluation/lessons/02-ground-truth.md', '01-agentic-rag/lessons/16-other-frameworks.md']
Q6 — ranked first: 01-agentic-rag/lessons/13-function-calling.md


**Answer Q6: `01-agentic-rag/lessons/13-function-calling.md`** — first in neither
list alone, but ranked high in both.